# Voting kết quả OCR của nhiều model

Gộp kết quả OCR của **ít nhất 3 model** cho cùng các trang (ví dụ PaddleOCR-VL, HunyuanOCR, API CLC của Lab) rồi bỏ
phiếu theo **từng chữ Hán**. Output của notebook này là **kết quả OCR cuối** của nhóm: dùng nó thay cho Output notebook
OCR ở mọi bước sau (dịch, `3_tao_input_test_chuoi`, `4_danh_gia`) và khi nộp bài.

**Chạy offline trên máy:** chọn kernel `.venv` đã cài rapidfuzz, opencc-python-reimplemented và tqdm.
Notebook chỉ đọc `data/ocr_raw/<sách>/*_shard*.jsonl` đã chuẩn hóa, không đọc các phiên OCR cũ.
Mở từ gốc dự án hoặc `data/` đều được. Output ghi vào **`data/voting/`**; chạy lại sẽ ghi đè kết quả cùng tên.
Nếu có `golden.jsonl` bên trong `data/`, notebook tự tìm để tính CER. Không cần mạng hay GPU.

**Cách voting của từng trang:**

1. Bỏ kết quả hỏng của model nào đó ở trang đó: trống, chạm trần token, hoặc lặp vòng.
2. Chọn **bản trục**: bản giống các bản còn lại nhất, tức tổng khoảng cách chỉnh sửa tới các bản khác nhỏ nhất. Hoà thì
   theo thứ tự `UU_TIEN`.
3. Gióng từng bản còn lại vào bản trục theo chữ Hán, bằng khoảng cách chỉnh sửa. So sánh không phân biệt giản thể với
   phồn thể.
4. Ở mỗi chữ của bản trục: chữ nào được nhiều model nhất thì giữ chữ đó; hoà thì giữ chữ của bản trục; nếu đa số model
   không có chữ này thì bỏ. Chữ mà bản trục thiếu chỉ được thêm vào khi đa số model cùng có.
5. Dấu câu và xuống dòng giữ theo bản trục.

Trang chỉ có 2 model thì không đủ đa số, lấy bản theo thứ tự `UU_TIEN`.

**Output đo lường** (ghi vào `data/voting/metrics/`):

- `page_metrics.jsonl`: mỗi trang gồm `voted_text`, `anchor_model`, `agreement_rate`, `total_chars`, `disagreement_count`, `selected_source`.
- `vote_distribution.jsonl`: phiếu của từng model tại từng vị trí, chữ được chọn và quy tắc chọn (kể cả chỗ chèn/xóa).
- `book_metrics.csv`: agreement trung bình và bình quân gia quyền, tổng ký tự, số vị trí không đồng thuận theo từng sách.
- `pages_cer.csv`, `cer_by_model.csv`: CER của từng model và của voting, số ký tự đúng/sai/chèn/xóa, số trang voting tốt hơn/kém hơn từng model (chỉ khi có `golden.jsonl`).
- `kiem_tay.csv`: tối đa 30 vị trí voting khác tham chiếu; cột `ben_dung` chỉ nhận `vote`, `tham_chieu`, `ca_hai_sai`, `khong_ro`.


In [1]:
UU_TIEN = [ "tonggu", "clc", "paddle", "ppocr", "hunyuan"]  # thứ tự ưu tiên khi hoà
# Tên thư mục OCR -> tên sách trong golden.jsonl (không sửa dữ liệu tham chiếu).
GOLD_BOOK_MAP = {
    "HDTTK": "Hoa_Đà_Trung_tàng_kinh",
    "ONDK": "Ôn_bệnh_điều_biện",
    **{f"VSYC_{i}": f"Vệ_sinh_yếu_chỉ_-_tập_{i}" for i in range(1, 7)},
}

def golden_id(book, page):
    return f"{GOLD_BOOK_MAP.get(book, book)}__p{page + 1}"


In [2]:
# Chạy offline: các thư viện cần được cài sẵn trong kernel đang chọn.
import importlib.util
required = {"rapidfuzz": "rapidfuzz", "opencc": "opencc-python-reimplemented",
            "tqdm": "tqdm"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ImportError("Kernel thiếu thư viện: " + ", ".join(missing) +
                      ". Hãy cài vào đúng môi trường Python trước khi chạy offline.")


In [3]:
# Dùng khi Input không có ocr_worker.py (ví dụ Input là zip Output tải về rồi tải lên lại)
OCR_WORKER_GON = r'''"""Bản rút gọn ocr_worker.py của notebook OCR mẫu: chỉ các hàm đọc kết quả OCR."""
from __future__ import annotations

import re


def md_to_lines(md: str) -> list[str]:
    """Markdown/HTML của OCR -> các dòng text thuần (bỏ #, **, ảnh, thẻ bảng, toạ độ hình)."""
    md = re.sub(r"</t[dh]>", " ", md)
    md = re.sub(r"</tr>|<br\s*/?>", "\n", md)
    md = re.sub(r"<[^>]+>", "", md)
    md = re.sub(r"!\[[^\]]*\]\([^)]*\)", "", md)
    out = []
    for ln in md.splitlines():
        ln = re.sub(r"^\s{0,3}#{1,6}\s*", "", ln)
        ln = re.sub(r"^\s*[-*+]\s+(?=\S)", "", ln)
        ln = re.sub(r"(?:[ \t,，]*\(\d{1,4},\d{1,4}\),\(\d{1,4},\d{1,4}\))+\s*$", "", ln)  # toạ độ hình của Hunyuan
        ln = ln.replace("**", "").replace("__", "").strip()
        if ln.startswith("|") and ln.endswith("|"):
            ln = " ".join(c.strip() for c in ln.strip("|").split("|") if c.strip())
        if ln and not re.fullmatch(r"[-|:\s]+", ln):
            out.append(ln)
    return out


def page_lines(rec: dict) -> list[str]:
    """1 trang OCR -> các dòng chữ theo thứ tự đọc."""
    if "blocks" not in rec:  # hunyuan: text markdown của cả trang
        return md_to_lines(rec["text"])
    ordered = sorted((b for b in rec["blocks"] if b["order"] is not None), key=lambda b: b["order"])
    center = lambda b: ((b["box"][0] + b["box"][2]) / 2, (b["box"][1] + b["box"][3]) / 2)  # noqa: E731
    for b in (b for b in rec["blocks"] if b["order"] is None):  # bảng, chú thích hình: paddle không xếp thứ tự đọc
        if not ordered:
            ordered.append(b)
            continue
        cx, cy = center(b)
        k = min(range(len(ordered)), key=lambda i: (center(ordered[i])[0] - cx) ** 2 + (center(ordered[i])[1] - cy) ** 2)
        ordered.insert(k + (cy > center(ordered[k])[1]), b)  # đặt cạnh vùng gần nhất
    return [ln for b in ordered for ln in md_to_lines(b["text"])]
'''

import csv, glob, json, re, shutil, sys, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from tqdm.auto import tqdm
from opencc import OpenCC
from rapidfuzz.distance import Levenshtein

# Tìm data/ khi kernel chạy từ gốc dự án hoặc từ bên trong data/.
DATA_DIR = next((base / "data" for base in [Path.cwd(), *Path.cwd().parents]
                 if (base / "data" / "ocr_raw").is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Không tìm thấy data/ocr_raw/. Hãy chạy từ thư mục dự án hoặc data/.")
INPUT_RAW = DATA_DIR / "ocr_raw"  # chỉ dùng dữ liệu đã chuẩn hóa, không quét các phiên OCR cũ
WORK = DATA_DIR / "voting"
RAW, TXT, MET = WORK / "ocr_raw", WORK / "txt", WORK / "metrics"
RAW.mkdir(parents=True, exist_ok=True)
TXT.mkdir(exist_ok=True)
MET.mkdir(exist_ok=True)
workers = []  # dùng bộ đọc rút gọn đi kèm, không phụ thuộc Input hay thư viện GPU
if workers:
    shutil.copy(workers[0], WORK / "ocr_worker.py")  # để notebook 3, 4 đọc được Output này
else:  # Input là zip Output tải về, không kèm ocr_worker.py: dùng bản rút gọn, chỉ có hàm đọc kết quả
    (WORK / "ocr_worker.py").write_text(OCR_WORKER_GON, encoding="utf-8")
sys.path.insert(0, str(WORK))
import ocr_worker as ow

# 1. Gom ocr_raw/ của mọi Input về chung một chỗ (giữ nguyên file của từng model)
input_files = sorted(INPUT_RAW.glob("*/*_shard*.jsonl"))
if not input_files:
    raise FileNotFoundError(f"Không có shard OCR trong {INPUT_RAW}")
print("Input:", INPUT_RAW, "| Output:", WORK)
for f in tqdm(input_files,
              desc="Gom ocr_raw", unit="file"):
    f = Path(f)
    if f.name.startswith("vote_") or "_layout_shard" in f.name:  # hunyuan_layout_*: vài trang vẽ layout, không phải OCR
        continue
    dst = RAW / f.parent.name / f.name
    # Ghi lại cùng tên khi chạy lại notebook, không nhân đôi shard.
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(f, dst)


def loop_ratio(t):
    t = re.sub(r"\s+", " ", t)
    if len(t) < 600:
        return 0.0
    ch = [t[i:i + 30] for i in range(0, len(t) - 30, 10)]
    return 1 - len(set(ch)) / len(ch)


trang = defaultdict(lambda: defaultdict(dict))  # sách -> trang (từ 0) -> engine -> chữ
hong = defaultdict(lambda: defaultdict(set))     # sách -> trang -> engine bị bỏ vì hỏng
for f in tqdm(input_files, desc="Đọc shard", unit="file"):
    eng = re.sub(r"_retry$", "", f.name.split("_shard")[0])
    for line in open(f, encoding="utf-8"):
        try:
            r = json.loads(line)
        except json.JSONDecodeError:
            continue
        t = "\n".join(ow.page_lines(r)).strip()
        if not t or r.get("hit_max") or loop_ratio(t) > 0.3:
            hong[f.parent.name][r["page"]].add(eng)
            continue
        trang[f.parent.name][r["page"]][eng] = t

HAN = re.compile(r"[㐀-鿿豈-﫿\U00020000-\U0003134f]")
cc = OpenCC("t2s")


def khoa(c):  # so 2 chữ không phân biệt giản / phồn
    return cc.convert(unicodedata.normalize("NFKC", c))


def rank(e):
    return UU_TIEN.index(e) if e in UU_TIEN else len(UU_TIEN)


def canh_hang(pk, seq, anchor, engines, bans):
    """Gióng mọi bản vào bản trục; trả phiếu / chữ thiếu / chỗ chèn của từng model."""
    n = len(pk)
    votes = [dict() for _ in range(n)]       # vị trí -> engine -> chữ (đã chuẩn hoá)
    missing = [set() for _ in range(n)]      # vị trí -> engine không có chữ
    inserts = defaultdict(dict)              # ranh giới -> engine -> chuỗi chèn
    insert_goc = {}                          # chuỗi chèn chuẩn hoá -> chữ gốc
    dang = [dict() for _ in range(n)]        # chữ chuẩn hoá -> chữ gốc (khi model đọc khác bản trục)
    for e in engines:
        s = seq[e]
        if e == anchor:
            for i, ch in enumerate(pk):
                votes[i][e] = ch
            continue
        goc = [c for c in bans[e] if HAN.match(c)]
        for tag, i1, i2, j1, j2 in Levenshtein.opcodes(pk, s):
            if tag == "equal":
                for k in range(i2 - i1):
                    votes[i1 + k][e] = pk[i1 + k]
            elif tag == "replace":
                c = min(i2 - i1, j2 - j1)
                for k in range(c):
                    votes[i1 + k][e] = s[j1 + k]
                    dang[i1 + k].setdefault(s[j1 + k], goc[j1 + k])
                for k in range(c, i2 - i1):
                    missing[i1 + k].add(e)
                if j2 - j1 > c:
                    w = s[j1 + c:j2]
                    inserts[i1 + c][e] = w
                    insert_goc.setdefault(w, "".join(goc[j1 + c:j2]))
            elif tag == "delete":
                for k in range(i1, i2):
                    missing[k].add(e)
            elif tag == "insert":
                w = s[j1:j2]
                inserts[i1][e] = w
                insert_goc.setdefault(w, "".join(goc[j1:j2]))
    for i in range(n):
        for e in engines:
            if e not in votes[i]:
                missing[i].add(e)
    return votes, missing, inserts, insert_goc, dang


def vote(bans):
    """bans: {engine: chữ của trang} -> (chữ sau voting, chi tiết để lưu metric)."""
    engines = list(bans)
    if len(engines) == 1:
        e = engines[0]
        text = bans[e]
        seq = [c for c in text if HAN.match(c)]
        n = len(seq)
        positions = [dict(position=i, kind="char", anchor_char=c, votes={e: c}, tally={c: 1}, missing=[],
                          chosen=c, source="single_model", source_models=[e])
                     for i, c in enumerate(seq)]
        detail = dict(anchor_model=e, agreement_rate=1.0, total_chars=n, disagreement_count=0,
                      positions=positions, selected_source={"rules": {"single_model": n}, "models": {e: n}})
        return text, detail

    seq = {e: "".join(khoa(c) for c in t if HAN.match(c)) for e, t in bans.items()}
    if len(engines) == 2:  # không đủ đa số: lấy bản theo UU_TIEN
        e = min(engines, key=rank)
        o = next(x for x in engines if x != e)
        pk = seq[e]
        votes, missing, inserts, _, _ = canh_hang(pk, seq, e, engines, bans)
        khop = sum(i2 - i1 for tag, i1, i2, _, _ in Levenshtein.opcodes(seq[e], seq[o]) if tag == "equal")
        n = len(pk)
        positions = []
        for i, ch in enumerate(pk):
            giong = len(votes[i]) == 2 and len(set(votes[i].values())) == 1
            positions.append(dict(position=i, kind="char", anchor_char=ch, votes=votes[i],
                                  tally=dict(Counter(votes[i].values())),
                                  missing=sorted(missing[i], key=rank), chosen=ch,
                                  source="both_agree" if giong else "anchor_two_models",
                                  source_models=sorted(votes[i], key=rank)))
        for j, d in sorted(inserts.items()):
            positions.append(dict(position=j, kind="insert", anchor_char=None, votes=d,
                                  tally=dict(Counter(d.values())), missing=[],
                                  chosen=None, source="ignored_two_models",
                                  source_models=sorted(d, key=rank)))
        detail = dict(anchor_model=e, agreement_rate=(khop / n if n else 1.0), total_chars=n,
                      disagreement_count=n - khop, positions=positions,
                      selected_source={"rules": {"anchor_two_models": n}, "models": {e: n}})
        return bans[e], detail

    truc = min(bans, key=lambda e: (sum(Levenshtein.distance(seq[e], seq[o]) for o in bans if o != e), rank(e)))
    P, pk = bans[truc], seq[truc]
    n, m = len(pk), len(bans)
    votes, missing, inserts, insert_goc, dang = canh_hang(pk, seq, truc, engines, bans)
    phieu = [Counter(votes[i].values()) for i in range(n)]
    bo = [m - len(votes[i]) for i in range(n)]

    def them_at(j):
        if not inserts.get(j):
            return None
        w, sl = Counter(inserts[j].values()).most_common(1)[0]
        if sl > m / 2:
            return dict(text=insert_goc.get(w, w), votes=dict(inserts[j]), source="insert_majority",
                        source_models=sorted([e for e, x in inserts[j].items() if x == w], key=rank))
        return None

    def chon(i, ch):
        top = max(phieu[i].values())
        if bo[i] > sum(phieu[i].values()):
            return None, "delete_majority", sorted(missing[i], key=rank)
        if phieu[i][pk[i]] == top:
            rule = "anchor_majority" if phieu[i][pk[i]] > m / 2 else "anchor_tie"
            return ch, rule, sorted([e for e, v in votes[i].items() if v == pk[i]], key=rank)
        w = max((k for k, v in phieu[i].items() if v == top), key=lambda k: k == pk[i])
        rule = "model_majority" if top > m / 2 else "model_plurality"
        return dang[i].get(w, w), rule, sorted([e for e, v in votes[i].items() if v == w], key=rank)

    out, positions, rules, models = [], [], Counter(), Counter()
    agreed, i = 0, 0

    def add_insert(j):
        ins = them_at(j)
        if not ins:
            return
        out.append(ins["text"])
        positions.append(dict(position=j, kind="insert", anchor_char=None, votes=ins["votes"],
                              tally=dict(Counter(ins["votes"].values())), missing=[],
                              chosen=ins["text"], source=ins["source"], source_models=ins["source_models"]))
        rules[ins["source"]] += 1
        for e in ins["source_models"]:
            models[e] += 1

    for ch in P:
        if not HAN.match(ch):
            out.append(ch)
            continue
        add_insert(i)
        chosen, rule, src = chon(i, ch)
        if chosen is not None:
            out.append(chosen)
        positions.append(dict(position=i, kind="char", anchor_char=ch, votes=votes[i],
                              tally=dict(Counter(votes[i].values())),
                              missing=sorted(missing[i], key=rank), chosen=chosen,
                              source=rule, source_models=src))
        rules[rule] += 1
        for e in src:
            models[e] += 1
        if phieu[i][pk[i]] == m:
            agreed += 1
        i += 1
    add_insert(n)
    detail = dict(anchor_model=truc, agreement_rate=(agreed / n if n else 1.0), total_chars=n,
                  disagreement_count=n - agreed, positions=positions,
                  selected_source={"rules": dict(rules), "models": dict(models)})
    return "".join(out), detail


# 2. Voting từng trang, ghi vote_shard0.jsonl, txt/ và metrics/
tong, page_metrics, vote_details, book_metrics = [], [], [], []
tong_trang = sum(len(v) for v in trang.values())
pbar = tqdm(total=tong_trang, desc="Voting", unit="trang")
for book in sorted(trang):
    recs, chua_du = [], 0
    for p in sorted(trang[book]):
        pbar.update(1)
        bans = trang[book][p]
        text, detail = vote(bans)
        chua_du += len(bans) < 3
        recs.append({"page": p, "text": text, "engines": sorted(bans), "truc": detail["anchor_model"],
                     "dong_y": detail["agreement_rate"], "bo_vi_hong": sorted(hong[book][p]),
                     "anchor_model": detail["anchor_model"], "agreement_rate": detail["agreement_rate"],
                     "total_chars": detail["total_chars"], "disagreement_count": detail["disagreement_count"],
                     "selected_source": detail["selected_source"]})
        page_metrics.append({"book_id": book, "page_id": p + 1, "page": p, "voted_text": text,
                             "engines": sorted(bans), "bo_vi_hong": sorted(hong[book][p]),
                             **{k: detail[k] for k in ("anchor_model", "agreement_rate", "total_chars",
                                                       "disagreement_count", "selected_source")}})
        vote_details.append({"book_id": book, "page_id": p + 1, "anchor_model": detail["anchor_model"],
                             "positions": detail["positions"]})

    with open(RAW / book / "vote_shard0.jsonl", "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in recs)
    (TXT / f"{book}.txt").write_text("\n\n".join(r["text"] for r in recs if r["text"]) + "\n", encoding="utf-8")

    engs = sorted({e for d in trang[book].values() for e in d})
    total = sum(r["total_chars"] for r in recs)
    dis = sum(r["disagreement_count"] for r in recs)
    rates = [r["agreement_rate"] for r in recs]
    rules, models = Counter(), Counter()
    for r in recs:
        rules.update(r["selected_source"]["rules"])
        models.update(r["selected_source"]["models"])
    book_metrics.append({"book_id": book, "pages": len(recs), "engines": ",".join(engs),
                         "models_count": len(engs), "pages_lt3_models": chua_du,
                         "anchor_models": json.dumps(dict(Counter(r["anchor_model"] for r in recs)), ensure_ascii=False),
                         "total_chars": total, "disagreement_count": dis,
                         "agreement_rate_mean": sum(rates) / len(rates) if rates else None,
                         "agreement_rate_weighted": 1 - dis / total if total else None,
                         "selected_source": json.dumps({"rules": dict(rules), "models": dict(models)}, ensure_ascii=False)})
    tong.append((book, len(recs), engs, chua_du, sum(rates) / len(rates) if rates else None))
pbar.close()

with open(MET / "page_metrics.jsonl", "w", encoding="utf-8") as f:
    f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in page_metrics)
with open(MET / "vote_distribution.jsonl", "w", encoding="utf-8") as f:
    f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in vote_details)
with open(MET / "book_metrics.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, list(book_metrics[0]) if book_metrics else
                       ["book_id", "pages", "engines", "models_count", "pages_lt3_models", "anchor_models",
                        "total_chars", "disagreement_count", "agreement_rate_mean",
                        "agreement_rate_weighted", "selected_source"])
    w.writeheader()
    w.writerows(book_metrics)

print(f"{'sách':<50}{'trang':>6}  {'model':<26}{'<3 model':>9}{'đồng ý':>8}")
for book, n, engs, chua_du, dy in tong:
    print(f"{book:<50}{n:>6}  {','.join(engs):<26}{chua_du:>9}{(f'{dy:.0%}' if dy is not None else '-'):>8}")
    if len(engs) < 3 or "clc" not in engs:
        print("   !!! chưa đúng yêu cầu: phải voting ít nhất 3 model, trong đó có CLC")
print("\n'đồng ý' = tỉ lệ chữ mà mọi model đọc giống nhau. Output: ocr_raw/ (đủ các model + vote), txt/ từ bản voting.")
print("Metric:", MET / "page_metrics.jsonl", "|", MET / "vote_distribution.jsonl", "|", MET / "book_metrics.csv")


/home/nthdayjpg/projects/chinese-vietnamese-vtm-corpus/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Input: /home/nthdayjpg/projects/chinese-vietnamese-vtm-corpus/data/ocr_raw | Output: /home/nthdayjpg/projects/chinese-vietnamese-vtm-corpus/data/voting


Voting: 100%|██████████| 1133/1133 [00:07<00:00, 145.69trang/s]


sách                                               trang  model                      <3 model  đồng ý
HDTTK                                                169  clc,hunyuan,paddle,ppocr,tonggu        0     89%
ONDK                                                 422  clc,hunyuan,paddle,ppocr          0     93%
VSYC_1                                               100  clc,hunyuan,paddle,ppocr,tonggu        0     45%
VSYC_2                                               103  clc,hunyuan,paddle,ppocr,tonggu        0     39%
VSYC_3                                               103  clc,hunyuan,paddle,ppocr,tonggu        0     39%
VSYC_4                                                70  clc,hunyuan,paddle,ppocr,tonggu        1     21%
VSYC_5                                                70  clc,hunyuan,paddle,ppocr,tonggu        1     21%
VSYC_6                                                96  clc,hunyuan,paddle,ppocr,tonggu        0     51%

'đồng ý' = tỉ lệ chữ mà mọi model đọc giống nh

In [4]:
# (Tuỳ chọn) CER của từng model, của voting và các thành phần đúng/sai/chèn/xóa, nếu có golden.jsonl
gj = sorted(str(p) for p in DATA_DIR.rglob("golden.jsonl") if WORK not in p.parents)
gold = {}


def norm(s):
    s = cc.convert(unicodedata.normalize("NFKC", s))
    return "".join(c for c in s if unicodedata.category(c)[0] not in "PSZC")


def edit_counts(ref, hyp):
    """Đúng / thay thế / chèn / xóa theo opcodes của Levenshtein (ref -> hyp)."""
    correct = sub = ins = dele = 0
    for tag, i1, i2, j1, j2 in Levenshtein.opcodes(ref, hyp):
        if tag == "equal":
            correct += i2 - i1
        elif tag == "replace":
            a, b, c = i2 - i1, j2 - j1, min(i2 - i1, j2 - j1)
            sub += c
            dele += a - c
            ins += b - c
        elif tag == "delete":
            dele += i2 - i1
        elif tag == "insert":
            ins += j2 - j1
    errors = sub + ins + dele
    return dict(ref_chars=len(ref), correct=correct, substitute=sub, insert=ins, delete=dele,
                errors=errors, cer=(errors / len(ref) if ref else 0.0))


if not gj:
    print("Không có golden.jsonl trong Input -> bỏ qua CER; vẫn giữ metrics/page_metrics.jsonl và book_metrics.csv")
else:
    if len(gj) > 1:
        print(f"Có {len(gj)} golden.jsonl, dùng file đầu tiên: {gj[0]}")
    gold = {r["id"]: r["zh"] for r in map(json.loads, open(gj[0], encoding="utf-8"))}
    vot = {b: {r["page"]: r["text"] for r in map(json.loads, open(RAW / b / "vote_shard0.jsonl", encoding="utf-8"))}
           for b in trang}
    page_cer_rows, cer_rows = [], []
    print(f"{'sách':<40}{'model':<12}{'trang':>6}{'CER':>8}{'đúng':>8}{'sai':>7}{'chèn':>7}{'xóa':>7}"
          f"{'vote tốt hơn':>13}{'vote kém hơn':>13}")

    for b in tqdm(sorted(trang), desc="Tính CER", unit="sách"):
        co = sorted({e for d in trang[b].values() for e in d})
        common = [p for p in sorted(trang[b])
                  if all(e in trang[b][p] for e in co)
                  and norm(gold.get(golden_id(b, p), ""))]
        per_page = {}
        for p in sorted(trang[b]):
            ref = norm(gold.get(golden_id(b, p), ""))
            if not ref:
                continue
            hyps = {e: trang[b][p][e] for e in co if e in trang[b][p]}
            hyps["vote"] = vot[b].get(p, "")
            for e, hyp in hyps.items():
                st = edit_counts(ref, norm(hyp))
                per_page[(p, e)] = st
                page_cer_rows.append(dict(book_id=b, page_id=p + 1, model=e, is_vote=(e == "vote"), **st))

        for e in co + ["vote"]:
            agg = Counter()
            for p in common:
                st = per_page.get((p, e))
                if st:
                    for k in ("ref_chars", "correct", "substitute", "insert", "delete", "errors"):
                        agg[k] += st[k]
            pages = sum(1 for p in common if (p, e) in per_page)
            better = worse = equal = 0
            if e != "vote":
                for p in common:
                    a, z = per_page.get((p, "vote")), per_page.get((p, e))
                    if not a or not z:
                        continue
                    if a["cer"] < z["cer"] - 1e-12:
                        better += 1
                    elif a["cer"] > z["cer"] + 1e-12:
                        worse += 1
                    else:
                        equal += 1
            row = dict(book_id=b, model=e, is_vote=(e == "vote"), pages=pages, **agg,
                       cer=(agg["errors"] / agg["ref_chars"] if agg["ref_chars"] else None),
                       pages_vote_better=better if e != "vote" else None,
                       pages_vote_worse=worse if e != "vote" else None,
                       pages_vote_equal=equal if e != "vote" else None)
            cer_rows.append(row)
            if pages:
                print(f"{b:<40}{e:<12}{pages:>6}{(row['cer'] if row['cer'] is not None else 0):>8.3f}"
                      f"{agg['correct']:>8}{agg['substitute']:>7}{agg['insert']:>7}{agg['delete']:>7}"
                      f"{(f'{better}/{worse}/{equal}' if e != 'vote' else '-'):>13}")

    with open(MET / "pages_cer.csv", "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, ["book_id", "page_id", "model", "is_vote", "ref_chars", "correct",
                               "substitute", "insert", "delete", "errors", "cer"])
        w.writeheader()
        w.writerows(page_cer_rows)
    with open(MET / "cer_by_model.csv", "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, ["book_id", "model", "is_vote", "pages", "ref_chars", "correct", "substitute",
                               "insert", "delete", "errors", "cer", "pages_vote_better",
                               "pages_vote_worse", "pages_vote_equal"])
        w.writeheader()
        w.writerows(cer_rows)
    print()
    print("Chỉ tổng hợp trên các trang mà mọi model của sách đều có kết quả và có golden, để so công bằng.")
    print("'vote tốt hơn / kém hơn / bằng' = số trang CER của voting thấp hơn / cao hơn / bằng CER của model đó.")
    print("Lưu ý: chữ Hán tham chiếu được đọc bằng HunyuanOCR nên HunyuanOCR có lợi thế khi so;")
    print("chỗ voting khác tham chiếu cần mở ảnh trang xem bên nào đúng, dùng kiem_tay.csv ở cell dưới.")
    print("Metric:", MET / "pages_cer.csv", "|", MET / "cer_by_model.csv")


sách                                    model        trang     CER    đúng    sai   chèn    xóa vote tốt hơn vote kém hơn


Tính CER:  12%|█▎        | 1/8 [00:00<00:01,  3.63sách/s]

HDTTK                                   clc            167   0.055   23154    622    259    453    132/16/19
HDTTK                                   hunyuan        167   0.008   24056    167     12      6     30/79/58
HDTTK                                   paddle         167   0.033   23572    531    132    126    118/10/39
HDTTK                                   ppocr          167   0.094   22925    864    979    440      165/0/2
HDTTK                                   tonggu         167   0.021   23777    399     45     53     72/19/76
HDTTK                                   vote           167   0.013   23934    254     13     41            -


Tính CER:  25%|██▌       | 2/8 [00:01<00:05,  1.04sách/s]

ONDK                                    clc            422   0.065  196619   7696   1670   4092      415/6/1
ONDK                                    hunyuan        422   0.004  207548    768     78     91  110/151/161
ONDK                                    paddle         422   0.011  206811   1180    726    416   201/65/156
ONDK                                    ppocr          422   0.027  204615   3008   1898    784    359/11/52
ONDK                                    vote           422   0.006  207566    719    433    122            -


Tính CER:  38%|███▊      | 3/8 [00:02<00:03,  1.47sách/s]

... (7 outputs left out)

Tính CER:  88%|████████▊ | 7/8 [00:03<00:00,  2.90sách/s]

VSYC_5                                  clc             18   0.666    3279   3837     74   2488       18/0/0
VSYC_5                                  hunyuan         18   0.873    2678   5687   1463   1239       18/0/0
VSYC_5                                  paddle          18   0.521    4741   2574    142   2289       15/3/0
VSYC_5                                  ppocr           18   0.527    4659   3274    112   1671       15/3/0
VSYC_5                                  tonggu          18   0.495    5105   3904    258    595       11/7/0
VSYC_5                                  vote            18   0.497    4914   2594     82   2096            -


Tính CER: 100%|██████████| 8/8 [00:03<00:00,  2.22sách/s]

VSYC_6                                  clc             94   0.216   24651   6293    167    305       94/0/0
VSYC_6                                  hunyuan         94   0.387   19879   9665    711   1705       94/0/0
VSYC_6                                  paddle          94   0.186   25615   5155    192    479       92/1/1
VSYC_6                                  ppocr           94   0.195   25228   5534     85    487       94/0/0
VSYC_6                                  tonggu          94   0.179   25787   5267    141    195       92/2/0
VSYC_6                                  vote            94   0.133   27175   3773     79    301            -

Chỉ tổng hợp trên các trang mà mọi model của sách đều có kết quả và có golden, để so công bằng.
'vote tốt hơn / kém hơn / bằng' = số trang CER của voting thấp hơn / cao hơn / bằng CER của model đó.
Lưu ý: chữ Hán tham chiếu được đọc bằng HunyuanOCR nên HunyuanOCR có lợi thế khi so;
chỗ voting khác tham chiếu cần mở ảnh trang xem bên nào đúng, 

In [5]:
# 4. Lấy ngẫu nhiên SO_CHO chỗ bản voting khác chữ tham chiếu -> kiem_tay.csv để nhóm mở ảnh trang kiểm tay
SO_CHO = 30
BEN_DUNG = ["vote", "tham_chieu", "ca_hai_sai", "khong_ro"]
import random

if gj and gold:
    def han(t):
        return [c for c in t if HAN.match(c)]

    def doi_vi_tri(a, b):  # ranh giới k trong a -> khoảng [lo, hi] trong b (gồm cả chữ b chèn thêm ở đó)
        lo, hi = [len(b)] * (len(a) + 1), [0] * (len(a) + 1)
        for tag, i1, i2, j1, j2 in Levenshtein.opcodes(a, b):
            for k in range(i1, i2 + 1):
                if tag == "insert":
                    x, y = j1, j2
                else:
                    x = y = j1 if tag == "delete" else (j2 if k == i2 else min(j1 + k - i1, j2))
                lo[k], hi[k] = min(lo[k], x), max(hi[k], y)
        return lo, hi

    cho = []
    for b in tqdm(sorted(trang), desc="Tìm chỗ lệch", unit="sách"):
        for p, text in vot[b].items():
            g = han(gold.get(golden_id(b, p), ""))
            v = han(text)
            if not g or not v:
                continue
            vk, gk = "".join(map(khoa, v)), "".join(map(khoa, g))
            ban = {e: han(t) for e, t in trang[b][p].items()}
            maps = {e: (doi_vi_tri(vk, "".join(map(khoa, s))), s) for e, s in ban.items()}
            for tag, i1, i2, j1, j2 in Levenshtein.opcodes(vk, gk):
                if tag == "equal" or max(i2 - i1, j2 - j1) > 4:
                    continue  # bỏ chỗ lệch dài, thường do đọc khác thứ tự
                model_outputs = {}
                for e, ((lo, hi), s) in maps.items():
                    model_outputs[e] = "".join(s[lo[i1]:hi[i2]]) or "(không có)"
                cho.append({
                    "book_id": b,
                    "page_id": p + 1,
                    "position": i1,
                    "reference_text": "".join(g[j1:j2]) or "(không có)",
                    "voted_text": "".join(v[i1:i2]) or "(không có)",
                    "model_outputs": json.dumps(model_outputs, ensure_ascii=False),
                    "ben_dung": "khong_ro",
                    "reason": "",
                    "context": "".join(v[max(0, i1 - 6):i1]) + "【" + "".join(v[i1:i2]) + "】" + "".join(v[i2:i2 + 6]),
                })
    random.seed(0)
    mau = random.sample(cho, min(SO_CHO, len(cho)))
    with open(WORK / "kiem_tay.csv", "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, ["book_id", "page_id", "position", "reference_text", "voted_text",
                               "model_outputs", "ben_dung", "reason", "context"], restval="")
        w.writeheader()
        w.writerows(sorted(mau, key=lambda r: (r["book_id"], r["page_id"], r["position"])))
    print(f"Có {len(cho)} chỗ bản voting khác chữ tham chiếu, đã lấy ngẫu nhiên {len(mau)} chỗ vào kiem_tay.csv.")
    print("ben_dung nhận đúng 4 giá trị:", ", ".join(BEN_DUNG))
    print("Mở ảnh từng trang, tìm đoạn ở cột context, sửa ben_dung từ 'khong_ro' thành kết luận và ghi reason.")
else:
    print("Không có golden.jsonl -> không tạo kiem_tay.csv")


Tìm chỗ lệch: 100%|██████████| 8/8 [00:05<00:00,  1.48sách/s]

Có 27485 chỗ bản voting khác chữ tham chiếu, đã lấy ngẫu nhiên 30 chỗ vào kiem_tay.csv.
ben_dung nhận đúng 4 giá trị: vote, tham_chieu, ca_hai_sai, khong_ro
Mở ảnh từng trang, tìm đoạn ở cột context, sửa ben_dung từ 'khong_ro' thành kết luận và ghi reason.


## 5. Kiểm tra `ben_dung` sau khi kiểm tay

Sau khi điền `kiem_tay.csv`, chạy ô dưới để chắc chắn cột `ben_dung` chỉ chứa bốn giá trị:
`vote`, `tham_chieu`, `ca_hai_sai`, `khong_ro`.


In [6]:
import csv
from collections import Counter

path = WORK / "kiem_tay.csv"
if not path.exists():
    print("Chưa có kiem_tay.csv (cần golden.jsonl và chạy ô 4).")
else:
    rows = list(csv.DictReader(open(path, encoding="utf-8-sig")))
    bad = sorted({r["ben_dung"].strip() for r in rows} - set(BEN_DUNG) - {""})
    print("Số dòng:", len(rows), "| ben_dung:", dict(Counter(r["ben_dung"] or "(trống)" for r in rows)))
    if bad:
        print("!!! Giá trị không hợp lệ:", bad, "-> chỉ dùng:", ", ".join(BEN_DUNG))
    else:
        print("OK: ben_dung chỉ dùng các giá trị hợp lệ (hoặc còn trống).")


Số dòng: 30 | ben_dung: {'khong_ro': 30}
OK: ben_dung chỉ dùng các giá trị hợp lệ (hoặc còn trống).
